In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt

In [14]:
# глобальные параметры
K_NEIGHBORS_USERS = 100
TOP_X_RECS = 100
TEST_SIZE = 0.2
SAMPLE_USER_ID = 2

In [3]:
# функция загрузки данных
def load_movielens_data():
    # Загружаем рейтинги (UserID::MovieID::Rating::Timestamp)
    ratings = pd.read_csv('ratings.dat', sep='::', engine='python',
                          names=['userId', 'movieId', 'rating', 'timestamp'],
                          encoding='ISO-8859-1')
    
    # Загружаем фильмы (MovieID::Title::Genres)
    movies = pd.read_csv('movies.dat', sep='::', engine='python',
                         names=['movieId', 'title', 'genres'],
                         encoding='ISO-8859-1')
    
    # Загружаем пользователей (UserID::Gender::Age::Occupation::Zip-code)
    users = pd.read_csv('users.dat', sep='::', engine='python',
                         names=['userId', 'gender', 'age', 'occupation', 'zip-code'],
                         encoding='ISO-8859-1')
    
    return ratings, movies, users

In [4]:
# функция создания разреженной матрицы (стр. пользователи/столб. фильм)
def create_user_movie_matrix(df):
    
    # pivot таблица: пользователи как строки, фильмы как столбцы
    pivot_table = df.pivot(index='userId', columns='movieId', values='rating').fillna(0)
    
    # преобразуем в разреженную матрицу
    sparse_matrix = csr_matrix(pivot_table.values)
    
    return pivot_table, sparse_matrix

In [5]:
# фунция получения рекомендаций для пользователя
def get_user_recommendations(user_id, n_recs=TOP_X_RECS):
    
    # проверяем, есть ли пользователь в обучающей выборке
    if user_id not in train_pivot_users.index:
        return f"Пользователь ID {user_id} не найден в обучающей выборке."
    
    # находим индекс пользователя в матрице
    user_idx = train_pivot_users.index.get_loc(user_id)
    
    # получаем вектор оценок пользователя
    user_vector = train_pivot_users.iloc[user_idx, :].values.reshape(1, -1)
    
    # находим K ближайших соседей (пользователей)
    distances, indices = model_knn_users.kneighbors(user_vector, n_neighbors=K_NEIGHBORS_USERS + 1)
    
    # собираем оценки соседей для каждого фильма
    neighbor_ratings = {}
    neighbor_weights = {}
    
    # добавляем нормализацию и отсечение выбросов
    similarities = []
    for i in range(1, len(distances.flatten())):
        # преобразуем расстояние в сходство (0..1, где 1 = идеально похожи)
        similarity = 1 - distances.flatten()[i]
        
        # отсекаем слишком низкое сходство (меньше 0.3)
        if similarity < 0.3:
            continue
            
        similarities.append(similarity)
    
    # если нет k похожих соседей, берем всех
    if len(similarities) < 3:
        # используем всех соседей, но с меньшим весом
        for i in range(1, len(distances.flatten())):
            similarity = max(0.1, 1 - distances.flatten()[i])
            similarities.append(similarity)
    
    # проходим по всем соседям
    for i in range(1, len(distances.flatten())):
        similarity = 1 - distances.flatten()[i]
        
        # пропускаем слишком непохожих соседей
        if similarity < 0.1:
            continue
            
        neighbor_idx = indices.flatten()[i]
        neighbor_id = train_pivot_users.index[neighbor_idx]
        
        # получаем оценки соседа
        neighbor_ratings_vec = train_pivot_users.iloc[neighbor_idx, :]
        
        # для каждого фильма который оценил сосед
        nonzero_ratings = neighbor_ratings_vec[neighbor_ratings_vec > 0]
        
        for movie_id, rating in nonzero_ratings.items():
            if movie_id not in neighbor_ratings:
                neighbor_ratings[movie_id] = 0
                neighbor_weights[movie_id] = 0
            
            # взвешенная сумма оценок
            neighbor_ratings[movie_id] += rating * similarity
            neighbor_weights[movie_id] += similarity
    
    # получаем оценки текущего пользователя
    user_ratings = train_pivot_users.iloc[user_idx, :]
    user_watched = user_ratings[user_ratings > 0].index.tolist()
    
    # добавляем средний рейтинг пользователя для нормализации
    user_avg_rating = user_ratings[user_ratings > 0].mean() if len(user_watched) > 0 else 3.0
    
    # вычисляем предсказанные оценки
    predictions = []
    for movie_id in neighbor_ratings:
        if movie_id not in user_watched and neighbor_weights[movie_id] > 0:
            # взвешенное среднее
            weighted_avg = neighbor_ratings[movie_id] / neighbor_weights[movie_id]
            
            # нормализация с учетом среднего пользователя, не даем оценкам уходить в крайности
            normalized_rating = weighted_avg * 0.8 + user_avg_rating * 0.2
            
            # ограничиваем диапазон 1-5
            final_rating = max(1.0, min(5.0, normalized_rating))
            
            predictions.append({
                'movieId': movie_id,
                'title': movie_titles.get(movie_id, "Unknown"),
                'predicted_rating': final_rating,
                'raw_score': weighted_avg  # Сохраняем сырую оценку для отладки
            })
    
    # сортируем по предсказанной оценке
    predictions.sort(key=lambda x: x['predicted_rating'], reverse=True)
    
    return predictions[:n_recs]

In [6]:
# функция оценки качества (RMSE)
def evaluate_model_user_based(test_df, train_pivot, model, k_neighbors):
    actual_ratings = []
    predicted_ratings = []
    
    # берем подвыборку из теста для ускорения (max 500 операций)
    test_sample = test_df.sample(min(500, len(test_df)), random_state=42)
    
    for _, row in test_sample.iterrows():
        user_id = int(row['userId'])
        movie_id = int(row['movieId'])
        actual_rating = row['rating']
        
        # проверяем, есть ли пользователь и фильм в обучающей матрице
        if user_id in train_pivot.index and movie_id in train_pivot.columns:
            # находим индекс пользователя
            user_idx = train_pivot.index.get_loc(user_id)
            
            # получаем вектор оценок пользователя
            user_vector = train_pivot.iloc[user_idx, :].values.reshape(1, -1)
            
            # находим K соседей
            distances, indices = model.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
            
            # собираем оценки соседей для данного фильма
            neighbor_ratings = []
            neighbor_similarities = []
            
            for i in range(1, len(distances.flatten())):
                neighbor_idx = indices.flatten()[i]
                similarity = 1 - distances.flatten()[i]
                
                # оценка соседа для данного фильма
                neighbor_rating = train_pivot.iloc[neighbor_idx, train_pivot.columns.get_loc(movie_id)]
                
                if neighbor_rating > 0:  # если сосед оценил этот фильм
                    neighbor_ratings.append(neighbor_rating)
                    neighbor_similarities.append(similarity)
            
            # предсказание: взвешенное среднее оценок соседей
            if neighbor_ratings:
                if sum(neighbor_similarities) > 0:
                    predicted = np.average(neighbor_ratings, weights=neighbor_similarities)
                else:
                    predicted = np.mean(neighbor_ratings)
                
                predicted_ratings.append(predicted)
                actual_ratings.append(actual_rating)
    
    if not actual_ratings:
        return None
    
    return sqrt(mean_squared_error(actual_ratings, predicted_ratings))

In [15]:
# основная программа
if __name__ == "__main__":
    print("User-based collaborative filtering")
    
    # загрузка данных
    try:
        ratings_df, movies_df, users_df = load_movielens_data()
        print(f"   - Рейтингов: {len(ratings_df)}")
        print(f"   - Пользователей: {ratings_df['userId'].nunique()}")
        print(f"   - Фильмов: {ratings_df['movieId'].nunique()}")
    except FileNotFoundError as e:
        print(f"Ошибка: {e}")
        print("возможно файлы ratings.dat, movies.dat, users.dat не находятся в текущей директории")
        exit()
    
    # создание словаря названий фильмов
    movie_titles = dict(zip(movies_df['movieId'], movies_df['title']))
    
    # разделение на train/test
    print("\n Разделение данных на train/test")
    train_data, test_data = train_test_split(ratings_df, test_size=TEST_SIZE, random_state=42)
    print(f"   - Train: {len(train_data)} рейтингов ({int((1-TEST_SIZE)*100)}%)")
    print(f"   - Test:  {len(test_data)} рейтингов ({int(TEST_SIZE*100)}%)")
    
    # создание матрицы пользователи × фильмы
    print("\n Матрица пользователи × фильмы")
    train_pivot_users, train_sparse_users = create_user_movie_matrix(train_data)
    
    # обучение модели KNN
    model_knn_users = NearestNeighbors(
        metric='cosine', 
        algorithm='brute', 
        n_neighbors=K_NEIGHBORS_USERS, 
        n_jobs=-1
    )
    model_knn_users.fit(train_sparse_users)
    print("Модель обучена")
    
    # получение рекомендаций для выбранного пользователя
    print(f"\n5. Рекомендации для пользователя ID={SAMPLE_USER_ID}:")
    print("-"*60)
    
    recommendations = get_user_recommendations(SAMPLE_USER_ID)
    
    if isinstance(recommendations, list):
        if len(recommendations) > 0:
            print(f"\nТоп-{min(TOP_X_RECS, len(recommendations))} рекомендаций:")
            for i, rec in enumerate(recommendations, 1):
                title_display = rec['title'][:50] if len(rec['title']) > 50 else rec['title']
                print(f"{i:2}. {title_display:50} (предв. оценка: {rec['predicted_rating']:.2f})")
        else:
            print("   Не найдено рекомендаций (возможно, пользователь оценил все фильмы или нет соседей)")
    else:
        print(f"   {recommendations}")
    
    # оценка качества модели
    print("ОЦЕНКА КАЧЕСТВА МОДЕЛИ")
    
    # проверяем, что есть достаточно данных для оценки
    if len(test_data) > 0:
        rmse = evaluate_model_user_based(test_data, train_pivot_users, model_knn_users, K_NEIGHBORS_USERS)
        
        if rmse:
            print(f"\n RMSE на тестовой выборке: {rmse:.4f}")
            print(f"  Интерпретация: модель ошибается в среднем на {rmse:.2f} балла (по 5-балльной шкале)")
            
            # дополнительная метрика: точность в диапазоне ±1 балл
            # можно добавить позже, если нужно
        else:
            print("\n Недостаточно пересечений в тестовой выборке для расчета RMSE")
            print("  Рекомендуется увеличить размер обучающей выборки или изменить параметры")
    else:
        print("\nТестовая выборка пуста")
    
    # доп информация
    print("Параметры системы")
    
    print(f"  - Количество соседей (K):      {K_NEIGHBORS_USERS}")
    print(f"  - Топ рекомендаций:            {TOP_X_RECS}")
    print(f"  - Размер тестовой выборки:     {TEST_SIZE*100}%")
    print(f"  - ID пользователя:             {SAMPLE_USER_ID}")
    print(f"  - Метрика расстояния:          косинусная")
    print(f"  - Тип рекомендаций:            User-based CF")

User-based collaborative filtering
   - Рейтингов: 1000209
   - Пользователей: 6040
   - Фильмов: 3706

 Разделение данных на train/test
   - Train: 800167 рейтингов (80%)
   - Test:  200042 рейтингов (20%)

 Матрица пользователи × фильмы
Модель обучена

5. Рекомендации для пользователя ID=2:
------------------------------------------------------------

Топ-100 рекомендаций:
 1. Nights of Cabiria (Le Notti di Cabiria) (1957)     (предв. оценка: 4.75)
 2. Perez Family, The (1995)                           (предв. оценка: 4.75)
 3. Deterrence (1998)                                  (предв. оценка: 4.75)
 4. Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)      (предв. оценка: 4.75)
 5. Bitter Sugar (Azucar Amargo) (1996)                (предв. оценка: 4.75)
 6. Shaggy D.A., The (1976)                            (предв. оценка: 4.75)
 7. Boys on the Side (1995)                            (предв. оценка: 4.75)
 8. Battleship Potemkin, The (Bronenosets Potyomkin) ( (предв. оценка: 4.75)
 9. De